<a href="https://colab.research.google.com/github/kevinl03/stochastic-spread-modeling/blob/migrate-statarb-work/statarb/cex_gbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StatArb LightGBM Pipeline
Cross-exchange spread z-score prediction across all coins × exchange pairs.

**Target:** z-score of `spread_pct` at `t+5` snapshots, rolling-normalised over 60 snapshots  
**Features:** lag-1..5 of spread, ticker mid/BA, orderbook imbalance, trade flow, funding rate, OI  
**Model:** single LightGBM; `coin` and `pair` as native categoricals

## 1. Imports & Config

In [1]:
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import gc

from huggingface_hub import get_token

HF_TOKEN = get_token()
assert HF_TOKEN, 'HF_TOKEN missing — run: huggingface-cli login'
print('HF auth ready')

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# ── paths — adjust if your layout differs ────────────────────────────────────
DATA_ROOT  = Path("./cex_dat")  # unused if USE_HF = True
USE_HF     = True               # ← add this
HF_REPO    = "SFU-fintech-AI/statarb-crypto-research"

OUTPUT_DIR = Path("./outputs_cex_gbm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── model / feature config ────────────────────────────────────────────────────
HORIZON       = 2    # snapshots ahead to predict (default was 5 min)
ZSCORE_WINDOW = 120   # rolling window for z-score normalisation (default was 60 min)
N_LAGS        = 5    # lag depth for every feature group (default was 5)
MIN_PERIODS   = 30   # min observations before z-score is emitted (default 20)

LGBM_PARAMS = {
    "objective":         "regression",
    "metric":            ["rmse", "mae"],
    "learning_rate":     0.05, #0.05
    "num_leaves":        350, #127
    "min_child_samples": 25, #50
    "feature_fraction":  0.8, #0.7
    "bagging_fraction":  0.9, #0.8
    "bagging_freq":      5, #5
    "lambda_l1":         0.01, #0.1
    "lambda_l2":         0.1, #0.1
    "verbosity":         -1, #-1
    "n_jobs":            -1, #-1
    "seed":              42, #42
}
NUM_BOOST_ROUND = 2000 #1000
EARLY_STOPPING  = 100 #50

Token loaded: hf_nubaY ...


In [2]:
# only keep payloads for these — set to False to skip a table entirely
LOAD_TABLES = {
    "spread_matrix": True,
    "ticker":        True,
    "orderbook":     True,
    "trades":        True,
    "funding_rate":  True,  # toggle off to save RAM if still crashing
    "open_interest": True,  # toggle off to save RAM if still crashing
}

TOP_EXCHANGES = ["binance", "bybit", "okx", "coinbase", "kraken"]  # drop smaller venues
TOP_FEATURES  = {
    "ticker":    ["mid", "spread_bps", "bid_volume", "ask_volume"],  # drop vwap, pct_change_24h
    "orderbook": ["imbalance", "slippage_bps"],                      # drop depth/vwap cols
    "trades":    ["buy_sell_ratio", "total_volume"],                  # drop count, price_mean
    "funding":   ["funding_rate"],                                    # drop next_rate, mark_price
    "oi":        ["oi_amount"],                                       # drop oi_value (mostly null)
}

## 2. Data Loading
Drops error rows (`error IS NOT NULL`) on load — these are failed ccxt fetches, not missing data.

In [3]:
SUBSETS = ["spread_matrix", "ticker", "orderbook", "trades", "funding_rate", "open_interest"]

"""
def load_parquet(split_dir: Path, name: str) -> pd.DataFrame:
    path = split_dir / f"{name}.parquet"
    if not path.exists():
        print(f"  [skip] {path.name} not found")
        return pd.DataFrame()
    df = pd.read_parquet(path)
    if "error" in df.columns:
        df = df[df["error"].isna()].drop(columns=["error"])
    return df

def load_split(split_dir: Path) -> dict[str, pd.DataFrame]:
    return {n: load_parquet(split_dir, n) for n in SUBSETS}

print("Loading splits …")
train_raw = load_split(DATA_ROOT / "1_train")
test_raw  = load_split(DATA_ROOT / "2_test")
val_raw   = load_split(DATA_ROOT / "3_val")

for split, raw in [("train", train_raw), ("test", test_raw), ("val", val_raw)]:
    print(f"\n{split}:")
    for k, v in raw.items():
        print(f"  {k:25s} {len(v):>10,} rows")
"""

# HuggingFace subset names per split
HF_SUBSETS = {
    "train": {
        "spread_matrix": "spread_matrix",
        "ticker":        "ticker",
        "orderbook":     "orderbook",
        "trades":        "trades",
        "funding_rate":  "funding_rate",
        "open_interest": "open_interest",
    },
    "test": {
        "spread_matrix": "test_spread_matrix",
        "ticker":        "test_ticker",
        "orderbook":     "test_orderbook",
        "trades":        "test_trades",
        "funding_rate":  "test_funding_rate",
        "open_interest": "test_open_interest",
    },
    "val": {
        "spread_matrix": "validation_spread_matrix",
        "ticker":        "validation_ticker",
        "orderbook":     "validation_orderbook",
        "trades":        "validation_trades",
        "funding_rate":  "validation_funding_rate",
        "open_interest": "validation_open_interest",
    },
}

"""
def load_split(split_name: str) -> dict[str, pd.DataFrame]:
    from datasets import load_dataset
    result = {}
    for local_name, hf_name in HF_SUBSETS[split_name].items():
        print(f"  loading {hf_name} …", end=" ")
        df = load_dataset(HF_REPO, hf_name, split="train", token=HF_TOKEN).to_pandas()
        if "error" in df.columns:
            df = df[df["error"].isna()].drop(columns=["error"])
        print(df.shape)
        result[local_name] = df
    return result
"""

def load_split(split_name: str) -> dict[str, pd.DataFrame]:
    from datasets import load_dataset
    result = {}
    for local_name, hf_name in HF_SUBSETS[split_name].items():
        if not LOAD_TABLES.get(local_name, True):
            print(f"  skipping {local_name}")
            result[local_name] = pd.DataFrame()
            continue

        print(f"  loading {hf_name} …", end=" ")
        ds = load_dataset(HF_REPO, hf_name, split="train", token=HF_TOKEN)
        drop = [c for c in ["run_id", "ts", "market", "symbol", "error"]
                if c in ds.column_names]
        if drop:
            ds = ds.remove_columns(drop)
        df = ds.to_pandas()
        del ds
        gc.collect()

        records = []
        for row in df.itertuples():
            try:
                p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
                snap  = row.snapshot_idx
                coin  = row.coin
                exch  = getattr(row, "exchange", None)

                if local_name == "spread_matrix":
                    # explode pairwise list immediately — no list objects stored
                    for pair in p["pairwise_spreads"]:
                        if pair["ex1"] in TOP_EXCHANGES and pair["ex2"] in TOP_EXCHANGES:
                            records.append((snap, coin, pair["ex1"], pair["ex2"],
                                            float(pair["spread_bps"]),
                                            float(pair["p1"]), float(pair["p2"])))
                elif local_name == "ticker" and exch in TOP_EXCHANGES:
                    records.append((snap, coin, exch,
                                    float(p.get("mid") or 0),
                                    float(p.get("spread_bps") or 0),
                                    float(p.get("bid_volume") or 0),
                                    float(p.get("ask_volume") or 0)))
                elif local_name == "orderbook" and exch in TOP_EXCHANGES:
                    records.append((snap, coin, exch,
                                    float(p.get("imbalance") or 0),
                                    float(p.get("slippage_bps") or 0)))
                elif local_name == "trades" and exch in TOP_EXCHANGES:
                    records.append((snap, coin, exch,
                                    float(p.get("buy_sell_ratio") or 0),
                                    float(p.get("total_volume") or 0)))
                elif local_name == "funding_rate" and exch in TOP_EXCHANGES:
                    records.append((snap, coin, exch,
                                    float(p.get("funding_rate") or 0)))
                elif local_name == "open_interest" and exch in TOP_EXCHANGES:
                    records.append((snap, coin, exch,
                                    float(p.get("open_interest_amount") or 0)))
            except Exception:
                continue

        del df
        gc.collect()

        # build typed dataframe directly from tuples — far cheaper than dict records
        cols = {
            "spread_matrix": ["snapshot_idx","coin","exchange_a","exchange_b","spread_bps","p1","p2"],
            "ticker":        ["snapshot_idx","coin","exchange","mid","spread_bps","bid_volume","ask_volume"],
            "orderbook":     ["snapshot_idx","coin","exchange","imbalance","slippage_bps"],
            "trades":        ["snapshot_idx","coin","exchange","buy_sell_ratio","total_volume"],
            "funding_rate":  ["snapshot_idx","coin","exchange","funding_rate"],
            "open_interest": ["snapshot_idx","coin","exchange","oi_amount"],
        }
        out = pd.DataFrame(records, columns=cols[local_name])

        # cast all floats to float32 immediately
        num_cols = out.select_dtypes(include="float64").columns
        out[num_cols] = out[num_cols].astype("float32")

        # cast ints to int32
        int_cols = out.select_dtypes(include="int64").columns
        out[int_cols] = out[int_cols].astype("int32")

        result[local_name] = out
        print(out.shape, f"  mem: {out.memory_usage(deep=True).sum()/1e6:.1f} MB")
        del records, out
        gc.collect()

    return result


train_raw = load_split("train")
test_raw  = load_split("test")
val_raw   = load_split("val")

  loading spread_matrix … (898626, 7)   mem: 159.9 MB
  loading ticker … (449430, 7)   mem: 57.1 MB
  loading orderbook … (7821, 5)   mem: 0.9 MB
  loading trades … (449503, 5)   mem: 53.5 MB
  loading funding_rate … 

funding_rate.parquet:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

(23851, 4)   mem: 2.0 MB
  loading open_interest … 

open_interest.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

(8434, 4)   mem: 0.7 MB
  loading validation_open_interest … 

validation/open_interest.parquet:   0%|          | 0.00/372k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

(8479, 4)   mem: 0.7 MB


## 3. Feature Engineering — Spread Matrix
This is the backbone. Every other feature table joins onto the `(snapshot_idx, coin, pair)` grain produced here.

- Parses `payload` JSON if columns aren't already flat
- Derives `spread_pct` from whatever price columns are present
- Computes rolling z-score **and** shifts it forward by `HORIZON` to form the target
- Adds lag-1..N of both `spread_pct` and `zscore`

In [4]:
def build_spread_features(sm: pd.DataFrame) -> pd.DataFrame:
    if sm.empty:
        return pd.DataFrame()

    sm = sm.copy()

    sm["pair"] = sm["exchange_a"] + "__" + sm["exchange_b"]
    sm = sm.sort_values(["coin", "pair", "snapshot_idx"]).reset_index(drop=True)

    # ── z-score per (coin, pair), target = z-score at t+HORIZON ─────────
    grp = sm.groupby(["coin", "pair"])["spread_bps"]
    roll_mean = grp.transform(lambda x: x.rolling(ZSCORE_WINDOW, min_periods=MIN_PERIODS).mean())
    roll_std  = grp.transform(lambda x: x.rolling(ZSCORE_WINDOW, min_periods=MIN_PERIODS).std())
    sm["zscore"] = (sm["spread_bps"] - roll_mean) / roll_std.replace(0, np.nan)
    sm["target"] = sm.groupby(["coin", "pair"])["zscore"].transform(lambda x: x.shift(-HORIZON))

    # ── lag features ──────────────────────────────────────────────────────
    for lag in range(1, N_LAGS + 1):
        sm[f"spread_bps_lag{lag}"] = grp.transform(lambda x, l=lag: x.shift(l))
        sm[f"zscore_lag{lag}"] = sm.groupby(["coin", "pair"])["zscore"].transform(
            lambda x, l=lag: x.shift(l)
        )

    return sm

## 4. Feature Engineering — Ticker
Derives mid-price and bid-ask spread (%), lags both, then pivots wide so each exchange becomes its own column set.

In [5]:
def build_ticker_features(tk: pd.DataFrame) -> pd.DataFrame:
  if tk.empty: return pd.DataFrame()
  tk = tk.sort_values(["coin", "exchange", "snapshot_idx"])
  feat_cols = ["mid", "spread_bps", "bid_volume", "ask_volume"]
  for col in feat_cols:
      for lag in range(1, N_LAGS + 1):
          tk[f"{col}_lag{lag}"] = tk.groupby(["coin", "exchange"])[col].transform(
              lambda x, l=lag: x.shift(l))
  lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
  wide = tk.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                        values=lag_cols, aggfunc="first")
  wide.columns = [f"tk_{col}_{exch}" for col, exch in wide.columns]
  wide.columns.name = None
  return wide.reset_index()


## 5. Feature Engineering — Orderbook
Computes top-5-level order book imbalance: `(bid_vol - ask_vol) / (bid_vol + ask_vol)`.  
Parses nested `bids`/`asks` lists from payload if volume columns aren't already present.

In [6]:
def build_orderbook_features(ob: pd.DataFrame) -> pd.DataFrame:
  if ob.empty: return pd.DataFrame()
  ob = ob.sort_values(["coin", "exchange", "snapshot_idx"])
  feat_cols = ["imbalance", "slippage_bps"]
  for col in feat_cols:
      for lag in range(1, N_LAGS + 1):
          ob[f"{col}_lag{lag}"] = ob.groupby(["coin", "exchange"])[col].transform(
              lambda x, l=lag: x.shift(l))
  lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
  wide = ob.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                        values=lag_cols, aggfunc="first")
  wide.columns = [f"ob_{col}_{exch}" for col, exch in wide.columns]
  wide.columns.name = None
  return wide.reset_index()


## 6. Feature Engineering — Trades
Net signed flow per snapshot: `sum(buy_vol) - sum(sell_vol)`.  
If `side` is missing, falls back to unsigned total volume.

In [7]:
def build_trades_features(tr: pd.DataFrame) -> pd.DataFrame:
  if tr.empty: return pd.DataFrame()
  tr = tr.sort_values(["coin", "exchange", "snapshot_idx"])
  feat_cols = ["buy_sell_ratio", "total_volume"]
  for col in feat_cols:
      for lag in range(1, N_LAGS + 1):
          tr[f"{col}_lag{lag}"] = tr.groupby(["coin", "exchange"])[col].transform(
              lambda x, l=lag: x.shift(l))
  lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
  wide = tr.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                        values=lag_cols, aggfunc="first")
  wide.columns = [f"tr_{col}_{exch}" for col, exch in wide.columns]
  wide.columns.name = None
  return wide.reset_index()


## 7. Feature Engineering — Funding Rate & Open Interest
Both are perp-only signals (NaN for spot venues). LightGBM handles NaN natively so no imputation needed.

In [8]:
def build_funding_features(fr: pd.DataFrame) -> pd.DataFrame:
    if fr.empty: return pd.DataFrame()
    fr = fr.sort_values(["coin", "exchange", "snapshot_idx"])
    feat_cols = ["funding_rate"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            fr[f"{col}_lag{lag}"] = fr.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l))
    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = fr.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"fr_{col}_{exch}" for col, exch in wide.columns]
    wide.columns.name = None
    return wide.reset_index()

def build_oi_features(oi: pd.DataFrame) -> pd.DataFrame:
    if oi.empty: return pd.DataFrame()
    oi = oi.sort_values(["coin", "exchange", "snapshot_idx"])
    feat_cols = ["oi_amount"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            oi[f"{col}_lag{lag}"] = oi.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l))
    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = oi.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"oi_{col}_{exch}" for col, exch in wide.columns]
    wide.columns.name = None
    return wide.reset_index()

## 8. Merge All Features
Left-join everything onto the spread base frame on `(snapshot_idx, coin)`.  
Missing auxiliary rows become NaN — handled by LightGBM natively.

In [9]:
def build_feature_matrix(raw: dict[str, pd.DataFrame]) -> pd.DataFrame:
    print("  spread …", end=" ")
    base = build_spread_features(raw["spread_matrix"])
    if base.empty:
        raise ValueError("spread_matrix empty or unparseable")
    print(f"{base.shape}")

    aux_builders = {
        "ticker":   (build_ticker_features,   raw["ticker"]),
        "orderbook":(build_orderbook_features, raw["orderbook"]),
        "trades":   (build_trades_features,    raw["trades"]),
        "funding":  (build_funding_features,   raw["funding_rate"]),
        "OI":       (build_oi_features,        raw["open_interest"]),
    }

    df = base.copy()
    for label, (fn, data) in aux_builders.items():
        aux = fn(data)
        if aux is not None and not aux.empty:
            df = df.merge(aux, on=["snapshot_idx", "coin"], how="left")
            print(f"  + {label:10s} → {df.shape}")

    return df

print("Building train …")
df_train = build_feature_matrix(train_raw)
print("\nBuilding test …")
df_test  = build_feature_matrix(test_raw)
print("\nBuilding val …")
df_val   = build_feature_matrix(val_raw)

Building train …
  spread … (898626, 20)
ticker cols: ['snapshot_idx', 'coin', 'tk_ask_volume_lag1_binance', 'tk_ask_volume_lag1_bybit', 'tk_ask_volume_lag1_coinbase', 'tk_ask_volume_lag1_kraken', 'tk_ask_volume_lag1_okx', 'tk_ask_volume_lag2_binance', 'tk_ask_volume_lag2_bybit', 'tk_ask_volume_lag2_coinbase', 'tk_ask_volume_lag2_kraken', 'tk_ask_volume_lag2_okx', 'tk_ask_volume_lag3_binance', 'tk_ask_volume_lag3_bybit', 'tk_ask_volume_lag3_coinbase', 'tk_ask_volume_lag3_kraken', 'tk_ask_volume_lag3_okx', 'tk_ask_volume_lag4_binance', 'tk_ask_volume_lag4_bybit', 'tk_ask_volume_lag4_coinbase', 'tk_ask_volume_lag4_kraken', 'tk_ask_volume_lag4_okx', 'tk_ask_volume_lag5_binance', 'tk_ask_volume_lag5_bybit', 'tk_ask_volume_lag5_coinbase', 'tk_ask_volume_lag5_kraken', 'tk_ask_volume_lag5_okx', 'tk_bid_volume_lag1_binance', 'tk_bid_volume_lag1_bybit', 'tk_bid_volume_lag1_coinbase', 'tk_bid_volume_lag1_kraken', 'tk_bid_volume_lag1_okx', 'tk_bid_volume_lag2_binance', 'tk_bid_volume_lag2_bybit',

## 9. Prepare LightGBM Datasets
- Drops rows where `target` is NaN (first `HORIZON` rows of each group, and z-score warmup rows)
- `coin` and `pair` are encoded as LightGBM native categoricals — no one-hot needed
- Test/val columns are re-aligned to train's column set (some exchanges may be absent in shorter splits)

In [10]:
ID_COLS = {"snapshot_idx", "exchange_a", "exchange_b", "spread_bps", "zscore", "target", "p1", "p2"}

def prepare_dataset(df: pd.DataFrame, reference_cols=None):
    df = df.dropna(subset=["target"]).copy()

    cat_cols = [c for c in ["coin", "pair"] if c in df.columns]
    for c in cat_cols:
        df[c] = df[c].astype("category")

    feat_cols = [c for c in df.columns if c not in ID_COLS]
    X = df[feat_cols].copy()
    y = df["target"].values

    if reference_cols is not None:
        X = X.reindex(columns=reference_cols)
        for c in cat_cols:
            if c in X.columns:
                X[c] = X[c].astype("category")

    return X, y, feat_cols, cat_cols

X_train, y_train, feat_cols, cat_cols = prepare_dataset(df_train)
X_test,  y_test,  _,         _        = prepare_dataset(df_test,  reference_cols=X_train.columns)
X_val,   y_val,   _,         _        = prepare_dataset(df_val,   reference_cols=X_train.columns)

print(f"Train : {X_train.shape}  |  target mean={y_train.mean():.3f}  std={y_train.std():.3f}")
print(f"Test  : {X_test.shape}")
print(f"Val   : {X_val.shape}")

Train : (891940, 182)  |  target mean=0.021  std=1.026
Test  : (785632, 182)
Val   : (310691, 182)


## 10. Train
Early stopping is evaluated on the **test** split.  
Once architecture is locked, consider using a time-based slice of train for early stopping instead, so test stays fully held-out.

In [11]:
dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
dtest  = lgb.Dataset(X_test,  label=y_test,  categorical_feature=cat_cols,
                     reference=dtrain, free_raw_data=False)

model = lgb.train(
    LGBM_PARAMS,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dtrain, dtest],
    valid_names=["train", "test"],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING, verbose=True),
        lgb.log_evaluation(100),
    ],
)
print(f"\nBest iteration: {model.best_iteration}")

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.959309	train's l1: 0.741872	test's rmse: 1.00772	test's l1: 0.769101
Early stopping, best iteration is:
[51]	train's rmse: 0.982315	train's l1: 0.758969	test's rmse: 1.007	test's l1: 0.768463

Best iteration: 51


## 11. Evaluate
`dir_acc` (directional accuracy) is the most operationally meaningful metric —  
it tells you how often the model correctly predicts whether the spread is above or below its rolling mean.

In [12]:
def evaluate(model, X, y, label):
    preds   = model.predict(X, num_iteration=model.best_iteration)
    mae     = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2      = r2_score(y, preds)
    dir_acc = np.mean(np.sign(preds) == np.sign(y))
    print(f"{label:20s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  DirAcc={dir_acc:.3%}")
    return {"label": label, "mae": mae, "rmse": rmse, "r2": r2, "dir_acc": dir_acc}

results = []
results.append(evaluate(model, X_train, y_train, "train"))
results.append(evaluate(model, X_test,  y_test,  "test"))
if len(y_val) > 0:
    results.append(evaluate(model, X_val, y_val, "val (partial)"))

results_df = pd.DataFrame(results)
results_df

train                 MAE=0.7590  RMSE=0.9823  R²=0.0829  DirAcc=59.789%
test                  MAE=0.7685  RMSE=1.0070  R²=0.0440  DirAcc=57.112%
val (partial)         MAE=0.7639  RMSE=1.0017  R²=0.0792  DirAcc=59.369%


,label,mae,rmse,r2,dir_acc
0,train,0.758969,0.982315,0.082950,0.597890
1,test,0.768463,1.007003,0.043982,0.571117
2,val (partial),0.763946,1.001739,0.079235,0.593686


## 12. Feature Importance

In [13]:
imp_df = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(imp_df.head(30).to_string())

                            feature     importance
0                       zscore_lag1  119819.476791
1                       zscore_lag2   78936.279850
2                       zscore_lag3   60214.570728
3                       zscore_lag4   45383.518848
4                              coin   42177.364202
5                       zscore_lag5   39400.495086
6                              pair   22670.062889
7                   spread_bps_lag1   15881.720552
8                   spread_bps_lag2   10078.928289
9                   spread_bps_lag5    8582.398296
10                  spread_bps_lag3    8399.587626
11                  spread_bps_lag4    5833.534885
12  tr_buy_sell_ratio_lag5_coinbase    4902.661695
13  tr_buy_sell_ratio_lag1_coinbase    4555.277797
14       tr_buy_sell_ratio_lag5_okx    4496.884909
15   tr_buy_sell_ratio_lag1_binance    4444.057117
16     tr_buy_sell_ratio_lag5_bybit    4339.048899
17  tr_buy_sell_ratio_lag3_coinbase    4129.186802
18   tr_buy_sell_ratio_lag4_bin

## 13. Save Outputs

In [14]:
model.save_model(str(OUTPUT_DIR / "statarb_lgbm.txt"))
imp_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "eval_results.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/")
print("  statarb_lgbm.txt")
print("  feature_importance.csv")
print("  eval_results.csv")

Saved to outputs/
  statarb_lgbm.txt
  feature_importance.csv
  eval_results.csv
